In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import json
import pandas as pd
import random
import datetime
from tqdm import tqdm
from utils import geo_to_mercator, mercator_to_geo

import cv2
# from h3.unstable import vect
# import h3.api.numpy_int as h3
import h3
import torch

In [34]:
resolution = "res8"

In [35]:
merged_res8 = pd.read_csv(f"merged_df_{resolution}.csv")

In [36]:
# How many satellite images are typically in the same cell?
merged_res8["taxon_id-fp"] =  merged_res8[["taxon_id", "fp"]].apply(lambda row: '_'.join(row.values.astype(str)), axis=1)
merged_res8["taxon_id-fp"].value_counts()

taxon_id-fp
42223_S2B_MSIL1C_20220526T165849_N0400_R069_T15RTP_20220526T204229/tci/1918_3377.png     642
6930_S2B_MSIL1C_20221015T192359_N0400_R099_T09UYR_20221015T213613/tci/1246_2779.png      588
12727_S2A_MSIL1C_20220209T155411_N0400_R054_T18TVK_20220209T205650/tci/2389_3106.png     348
132473_S2A_MSIL1C_20221227T170721_N0509_R069_T14RPR_20221227T190330/tci/1883_3441.png    347
472766_S2A_MSIL1C_20220103T092401_N0301_R093_T34SFJ_20220103T104450/tci/4605_3112.png    279
                                                                                        ... 
158615_S2B_MSIL1C_20220514T112109_N0400_R037_T30UWC_20220514T120942/tci/4045_2720.png      1
53414_S2B_MSIL1C_20220514T112109_N0400_R037_T30UWC_20220514T120942/tci/4045_2720.png       1
56535_S2B_MSIL1C_20220514T112109_N0400_R037_T30UWC_20220514T120942/tci/4045_2720.png       1
791928_S2B_MSIL1C_20220514T112109_N0400_R037_T30UWC_20220514T120942/tci/4045_2720.png      1
53021_S2B_MSIL1C_20220514T112109_N0400_R037_T30UWC_2022051

In [37]:
np.sum(merged_res8["taxon_id-fp"].value_counts()==1), np.sum(merged_res8["taxon_id-fp"].value_counts()>1)

(702317, 161831)

In [4]:
sat_sinr_taxon_ids = np.unique(merged_res8["taxon_id"].values)

In [ ]:
wiki_data = torch.load("data/wiki_data_v4.pt")  # see LE-SINR paper (https://github.com/cvl-umass/le-sinr)

In [6]:
class_ids = [x[0] for x in wiki_data["keys"]]
len(np.unique(class_ids))

37889

In [7]:
wiki_data.keys()

dict_keys(['taxon_id', 'keys', 'data'])

In [8]:
wiki_data["data"].shape, wiki_data["taxon_id"].shape, len(wiki_data["keys"])

(torch.Size([127484, 4096]), torch.Size([37889]), 127484)

In [9]:
len(sat_sinr_taxon_ids)

40161

In [10]:

wiki_taxon_ids = wiki_data["taxon_id"].detach().numpy()
ctr = 0
non_ctr = 0
common_taxon_ids = []
for taxon_id in wiki_taxon_ids:
    if taxon_id in sat_sinr_taxon_ids:
        ctr += 1
        common_taxon_ids.append(taxon_id)
    else:
        non_ctr += 1
ctr, non_ctr

(32690, 5199)

In [11]:
# # how many text embeddings are there that are in the common_taxon_ids?
# keys_ctr = 0
# found_keys = []
# for entry in wiki_data["keys"]:
#     if entry[0] in common_taxon_ids:
#         keys_ctr += 1
#         found_keys.append(entry[1])

In [12]:
# make df with data_idx (index in wiki_data["data"])
# columns: text_emb_idx, taxon_id, section_name
wiki_data_df = []
wiki_taxon_ids = wiki_data["taxon_id"].detach().numpy()
for emb_idx, (class_id, section_name) in enumerate(wiki_data["keys"]):
    wiki_data_df.append([emb_idx, wiki_taxon_ids[class_id], section_name])
wiki_df = pd.DataFrame(wiki_data_df, columns=["text_emb_idx", "taxon_id", "section_name"])

In [13]:
merged_df = wiki_df.merge(merged_res8, how = 'inner', on = ['taxon_id'])

In [14]:
merged_df.shape

(6766647, 28)

In [15]:
merged_df.head()

,text_emb_idx,taxon_id,section_name,fp,datetime,year_x,month,day,col,row,...,observer_id,latitude_y,longitude_y,observed_on,year_y,taxon_id-lat-lon,h3_res6_y,h3_res7_y,h3_res9_y,taxon_id-h3_res8
0,1,18984,text,S2A_MSIL1C_20220510T160521_N0400_R054_T16PHS_2...,20220510T160521,2022,5,10,2180,3869,...,824100,9.89933,-84.177277,2019-05-15,2019,18984.0_9.8993296988_-84.1772774067,866d6930fffffff,876d69309ffffff,896d6930907ffff,18984_886d693091fffff
1,2,18984,Description,S2A_MSIL1C_20220510T160521_N0400_R054_T16PHS_2...,20220510T160521,2022,5,10,2180,3869,...,824100,9.89933,-84.177277,2019-05-15,2019,18984.0_9.8993296988_-84.1772774067,866d6930fffffff,876d69309ffffff,896d6930907ffff,18984_886d693091fffff
2,3,18984,Taxonomy,S2A_MSIL1C_20220510T160521_N0400_R054_T16PHS_2...,20220510T160521,2022,5,10,2180,3869,...,824100,9.89933,-84.177277,2019-05-15,2019,18984.0_9.8993296988_-84.1772774067,866d6930fffffff,876d69309ffffff,896d6930907ffff,18984_886d693091fffff
3,4,18984,Range and habitat,S2A_MSIL1C_20220510T160521_N0400_R054_T16PHS_2...,20220510T160521,2022,5,10,2180,3869,...,824100,9.89933,-84.177277,2019-05-15,2019,18984.0_9.8993296988_-84.1772774067,866d6930fffffff,876d69309ffffff,896d6930907ffff,18984_886d693091fffff
4,5,18984,In captivity,S2A_MSIL1C_20220510T160521_N0400_R054_T16PHS_2...,20220510T160521,2022,5,10,2180,3869,...,824100,9.89933,-84.177277,2019-05-15,2019,18984.0_9.8993296988_-84.1772774067,866d6930fffffff,876d69309ffffff,896d6930907ffff,18984_886d693091fffff


In [16]:
merged_df.columns

Index(['text_emb_idx', 'taxon_id', 'section_name', 'fp', 'datetime', 'year_x',
       'month', 'day', 'col', 'row', 'lon-lat', 'longitude_x', 'latitude_x',
       'h3_res6_x', 'h3_res7_x', 'h3_res8', 'h3_res9_x', 'observation_uuid',
       'observer_id', 'latitude_y', 'longitude_y', 'observed_on', 'year_y',
       'taxon_id-lat-lon', 'h3_res6_y', 'h3_res7_y', 'h3_res9_y',
       'taxon_id-h3_res8'],
      dtype='object')

In [17]:
merged_df["section_name"].value_counts()

section_name
text                              1259704
Description                        903112
Distribution and habitat           387976
Taxonomy                           369754
Distribution                       238361
                                   ...   
Proposed conservation measures          1
Relevant literature                     1
Subspecies and Distribution             1
Diet and Sound                          1
Types of Vocalization                   1
Name: count, Length: 5219, dtype: int64

In [18]:
merged_df["taxon_id"].value_counts()

taxon_id
48662      70704
47219      68835
6930       60112
13858      32202
7089       31272
           ...  
13791          1
1359763        1
16175          1
13417          1
465229         1
Name: count, Length: 32690, dtype: int64

In [19]:
merged_df.columns

Index(['text_emb_idx', 'taxon_id', 'section_name', 'fp', 'datetime', 'year_x',
       'month', 'day', 'col', 'row', 'lon-lat', 'longitude_x', 'latitude_x',
       'h3_res6_x', 'h3_res7_x', 'h3_res8', 'h3_res9_x', 'observation_uuid',
       'observer_id', 'latitude_y', 'longitude_y', 'observed_on', 'year_y',
       'taxon_id-lat-lon', 'h3_res6_y', 'h3_res7_y', 'h3_res9_y',
       'taxon_id-h3_res8'],
      dtype='object')

In [20]:
merged_df[f"taxon_id-h3_{resolution}"].value_counts()

taxon_id-h3_res8
42223_88446dc8cdfffff      5136
6930_8828d9c85dfffff       4704
132473_8848954243fffff     3123
48662_882ab250c7fffff      2640
12727_882a137b15fffff      2436
                           ... 
1372751_882a3349d7fffff       1
1372751_882b002a61fffff       1
1372751_882b8320bbfffff       1
13791_8854ad5b6dfffff         1
1372751_882a1545e9fffff       1
Name: count, Length: 835380, dtype: int64

In [ ]:
merged_df.to_csv(f"merged_df_{resolution}_withtextidx.csv", index=False)

In [24]:
merged_df.shape

(981901, 28)